# Session 2A — LangChain Agent

**GOAL:** Replace the messy while-loop from Session 1B with LangChain's clean agent framework.

Compare: ~120 lines of manual JSON parsing in 1B → ~40 lines of clean LangChain code here.

In [ ]:
import os, sys, warnings, logging

warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

try:
    import utils.dns_patch
except Exception:
    pass

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from utils.tools import fetch_emails, schedule_meeting

print('Setup complete.')

### Run the LangChain Agent

ONE LINE (`create_react_agent`) replaces the entire while-loop from Demo 1B!

In [ ]:
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', temperature=0)
tools = [fetch_emails, schedule_meeting]
agent = create_react_agent(llm, tools)

print('Running LangChain ReAct agent ...\n')

result = agent.invoke({
    'messages': [
        HumanMessage(content=(
            'Fetch my recent emails, analyze them, categorize each as '
            'urgent/meeting/task/info, and if you find any meeting '
            'requests, schedule them on my Google Calendar. '
            'Give me a full summary at the end.'
        ))
    ]
})

print('FULL AGENT CONVERSATION:')
print('-' * 50)
for msg in result['messages']:
    role = msg.type.upper()
    content = getattr(msg, 'content', '')
    if isinstance(content, list):
        content = '\n'.join(c.get('text', '') if isinstance(c, dict) and 'text' in c else str(c) for c in content)
    if content:
        print(f'\n[{role}]:')
        print(f'  {content}')
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f'  Tool Call: {tc["name"]}({tc["args"]})')

print('\nKEY TAKEAWAY:')
print('   Same result as 1B, but LangChain handled ALL the complexity.')
print('   The @tool decorator + create_react_agent() = Production-ready agent.')